# AA1 Competition – Preprocessing

This notebook covers the full preprocessing pipeline for the Wi-Fi CSI localization dataset:
1. Loading and basic inspection
2. Dealing with zero-variance (guard/pilot) subcarriers
3. Feature engineering: amplitude & phase from raw I/Q
4. Inter-antenna features (phase difference, amplitude ratio)
5. Dropping non-informative features (`seq_ctrl`)
6. Normalization with `StandardScaler`
7. Saving the preprocessed datasets

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import pickle
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.precision', 3)
pd.set_option('display.max_columns', 20)

## 1. Loading the data

In [ ]:
# The CSV uses semicolons as delimiters and the first column is the row ID
train = pd.read_csv('train_nt.csv', sep=';', index_col=0)
test  = pd.read_csv('test_nolabels_nt.csv', sep=';', index_col=0)

print(f'Train shape: {train.shape}')  # (12888, 261)
print(f'Test shape:  {test.shape}')   # (3223, 260)

## 2. Basic inspection

In [ ]:
train.head(3)

In [ ]:
# Data types
train.dtypes.value_counts()

In [ ]:
# Missing values
print('Train NaN:', train.isnull().sum().sum())
print('Test  NaN:', test.isnull().sum().sum())
# Dataset is clean — no missing values (controlled lab acquisition)

In [ ]:
# Target distribution
fig, ax = plt.subplots(figsize=(7, 3))
counts = train['position'].value_counts().sort_index()
ax.bar(counts.index, counts.values, color='steelblue', edgecolor='white')
ax.set_xlabel('Position (class)')
ax.set_ylabel('Sample count')
ax.set_title('Class distribution (train set)')
ax.set_xticks(range(10))
for i, v in enumerate(counts.values):
    ax.text(i, v + 5, str(v), ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('fig_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(counts)

In [ ]:
# Quick look at the non-CSI features
train[['seq_ctrl', 'aoa', 'rssi1', 'rssi2']].describe()

**Observations:**
- No missing values.
- Dataset is **well balanced** (~1200–1488 samples per class).
- `aoa` has a very large std (~3087) — the values are not in radians but are raw integer phase differences. They are very noisy and not well-separated per class, but still carry some signal.
- `rssi1` / `rssi2` range from −97 to −58 dBm. The 2-metre positions (1–4) have higher RSSI than the 5-metre ones (5–9), as expected.

## 3. Identifying active vs guard/pilot subcarriers

In IEEE 802.11 (OFDM), not all 64 subcarriers carry data:
- **Subcarrier 0**: DC subcarrier (always zero).
- **Subcarriers 27–37**: guard band subcarriers (unused).
- The remaining **52 subcarriers** are the active data + pilot subcarriers.

Columns with zero variance across all samples carry no information and must be dropped.

In [ ]:
# Find zero-variance subcarrier indices
zero_sub = []
active_sub = []
for n in range(64):
    col = f'I{n}_1'  # check antenna 1 I-component; both antennas share same pattern
    if train[col].std() == 0:
        zero_sub.append(n)
    else:
        active_sub.append(n)

print(f'Zero-variance (guard/DC) subcarrier indices: {zero_sub}')
print(f'Active subcarrier indices ({len(active_sub)}): {active_sub}')

In [ ]:
# Visualize mean amplitude per subcarrier per class to confirm
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ant_idx, ant in enumerate([1, 2]):
    ax = axes[ant_idx]
    for pos in range(10):
        subset = train[train['position'] == pos]
        amps = []
        for n in range(64):
            I = subset[f'I{n}_{ant}'].values
            Q = subset[f'Q{n}_{ant}'].values
            amps.append(np.sqrt(I**2 + Q**2).mean())
        ax.plot(range(64), amps, alpha=0.7, label=f'pos {pos}')
    ax.set_xlabel('Subcarrier index')
    ax.set_ylabel('Mean CSI amplitude')
    ax.set_title(f'Antenna {ant}')
    ax.legend(fontsize=6, ncol=2)
plt.suptitle('Mean CSI amplitude per subcarrier and position')
plt.tight_layout()
plt.savefig('fig_csi_amplitude_profile.png', dpi=150, bbox_inches='tight')
plt.show()

The plot confirms subcarriers 0 and 27–37 are zero (DC + guard band). The active subcarriers show **clearly different profiles per position**, validating that the CSI carries discriminative information.

## 4. Separating features and target

In [ ]:
X_train_raw = train.drop(columns=['position'])
y_train      = train['position'].astype(int)
X_test_raw   = test.copy()

print('X_train_raw:', X_train_raw.shape)
print('y_train:    ', y_train.shape)
print('X_test_raw: ', X_test_raw.shape)

## 5. Dropping non-informative features

### 5.1 `seq_ctrl` — sequence number
`seq_ctrl` is a uint16 packet sequence counter. It has no spatial meaning and will vary with timing, not position. It is dropped.

### 5.2 Zero-variance subcarrier columns
All I/Q columns for guard/DC subcarriers are identically zero — they carry no information and would only add noise to distance-based models.

In [ ]:
def drop_non_informative(df, zero_subcarriers):
    """Drop seq_ctrl and all I/Q columns for zero-variance subcarriers."""
    cols_to_drop = ['seq_ctrl']
    for n in zero_subcarriers:
        for ant in [1, 2]:
            cols_to_drop += [f'I{n}_{ant}', f'Q{n}_{ant}']
    # Only drop columns that exist in the dataframe
    cols_to_drop = [c for c in cols_to_drop if c in df.columns]
    return df.drop(columns=cols_to_drop)

X_train_clean = drop_non_informative(X_train_raw, zero_sub)
X_test_clean  = drop_non_informative(X_test_raw,  zero_sub)

print(f'After dropping: X_train {X_train_clean.shape}, X_test {X_test_clean.shape}')
# Remaining: aoa, rssi1, rssi2 + 52 active subcarriers × 2 antennas × 2 (I,Q) = 208 IQ cols → 211 total

## 6. Feature engineering

Raw I and Q components are not the most physically meaningful representation. We derive:

- **Amplitude** of each subcarrier: $A_n = \sqrt{I_n^2 + Q_n^2}$ — proportional to signal strength on that subcarrier.
- **Phase** of each subcarrier: $\phi_n = \arctan2(Q_n, I_n)$ — encodes multipath propagation delay.
- **Inter-antenna phase difference**: $\Delta\phi_n = \phi_n^{(1)} - \phi_n^{(2)}$ — related to the angle-of-arrival (this is what `aoa` approximates at a coarse level).
- **Amplitude ratio** between antennas: $R_n = A_n^{(1)} / (A_n^{(2)} + \epsilon)$ — captures relative fading between antennas.

In [ ]:
def engineer_csi_features(df, active_subcarriers):
    """
    From raw I/Q columns, compute amplitude, phase (per antenna),
    inter-antenna phase difference, and amplitude ratio.
    Returns a new DataFrame keeping aoa, rssi1, rssi2 plus all derived features.
    Drops the raw I/Q columns.
    """
    new_df = pd.DataFrame(index=df.index)

    # Keep non-IQ features as-is
    for col in ['aoa', 'rssi1', 'rssi2']:
        if col in df.columns:
            new_df[col] = df[col]

    eps = 1e-6  # avoid division by zero

    for n in active_subcarriers:
        I1 = df[f'I{n}_1'].values.astype(float)
        Q1 = df[f'Q{n}_1'].values.astype(float)
        I2 = df[f'I{n}_2'].values.astype(float)
        Q2 = df[f'Q{n}_2'].values.astype(float)

        amp1 = np.sqrt(I1**2 + Q1**2)
        amp2 = np.sqrt(I2**2 + Q2**2)
        phase1 = np.arctan2(Q1, I1)         # in [-pi, pi]
        phase2 = np.arctan2(Q2, I2)

        new_df[f'amp{n}_1']    = amp1
        new_df[f'amp{n}_2']    = amp2
        new_df[f'phase{n}_1']  = phase1
        new_df[f'phase{n}_2']  = phase2
        new_df[f'dphi{n}']     = phase1 - phase2        # inter-antenna phase diff
        new_df[f'aratio{n}']   = amp1 / (amp2 + eps)   # amplitude ratio ant1/ant2

    return new_df


X_train_eng = engineer_csi_features(X_train_clean, active_sub)
X_test_eng  = engineer_csi_features(X_test_clean,  active_sub)

print(f'Engineered features: {X_train_eng.shape[1]}')
print('Feature groups:')
print(f'  aoa, rssi1, rssi2        : 3')
print(f'  amp_1, amp_2 per subcarr : {len(active_sub)*2}')
print(f'  phase_1, phase_2         : {len(active_sub)*2}')
print(f'  dphi (phase diff)        : {len(active_sub)}')
print(f'  aratio (amp ratio)       : {len(active_sub)}')
print(f'  Total                    : {3 + len(active_sub)*6}')
X_train_eng.head(3)

In [ ]:
# Sanity check: no NaNs or Infs after feature engineering
print('NaN in train:', X_train_eng.isnull().sum().sum())
print('Inf in train:', np.isinf(X_train_eng.values).sum())
print('NaN in test: ', X_test_eng.isnull().sum().sum())
print('Inf in test: ', np.isinf(X_test_eng.values).sum())

## 7. Exploratory analysis on engineered features

In [ ]:
# RSSI per class — shows 2m vs 5m separation
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, feat in zip(axes, ['rssi1', 'rssi2']):
    data_per_class = [X_train_eng.loc[y_train == c, feat].values for c in range(10)]
    ax.boxplot(data_per_class, labels=range(10), patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.6))
    ax.set_xlabel('Position (class)')
    ax.set_ylabel('RSSI (dBm)')
    ax.set_title(feat)
    ax.grid(axis='y', alpha=0.3)
plt.suptitle('RSSI distribution per class\n(classes 0-4: 2m; classes 5-9: 5m)')
plt.tight_layout()
plt.savefig('fig_rssi_per_class.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Inter-antenna phase difference (dphi) for a few subcarriers per class
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
sample_subs = [1, 5, 10, 20, 45, 60]
for ax, n in zip(axes.flat, sample_subs):
    if n in active_sub:
        data = [X_train_eng.loc[y_train == c, f'dphi{n}'].values for c in range(10)]
        ax.boxplot(data, labels=range(10), patch_artist=True,
                   boxprops=dict(facecolor='coral', alpha=0.6))
        ax.set_title(f'dphi subcarrier {n}')
        ax.set_xlabel('Position')
        ax.set_ylabel('Phase diff (rad)')
        ax.grid(axis='y', alpha=0.3)
plt.suptitle('Inter-antenna phase difference per class for selected subcarriers')
plt.tight_layout()
plt.savefig('fig_dphi_per_class.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation of all engineered features with the target
corr_with_target = X_train_eng.copy()
corr_with_target['position'] = y_train.values
target_corr = corr_with_target.corr()['position'].drop('position').abs().sort_values(ascending=False)

print('Top 20 features by |correlation| with target:')
print(target_corr.head(20))

# Plot top 30
fig, ax = plt.subplots(figsize=(10, 5))
top30 = target_corr.head(30)
colors = ['#2196F3' if 'rssi' in n or 'aoa' in n else
          '#FF5722' if 'dphi' in n or 'aratio' in n else
          '#4CAF50' if 'amp' in n else '#9C27B0'
          for n in top30.index]
ax.barh(top30.index[::-1], top30.values[::-1], color=colors[::-1])
ax.set_xlabel('|Pearson correlation| with position')
ax.set_title('Top 30 features by correlation with target')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('fig_feature_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Normalization

Many classifiers (SVM, KNN, logistic regression) are sensitive to feature scale. We apply **StandardScaler** (zero mean, unit variance), fitted **only on the training set** and then applied to the test set to avoid data leakage.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_eng)   # fit + transform on train
X_test_scaled  = scaler.transform(X_test_eng)        # only transform on test

# Wrap back as DataFrames for convenience
feature_names = X_train_eng.columns.tolist()
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_names, index=X_train_eng.index)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=feature_names, index=X_test_eng.index)

print('X_train_scaled:', X_train_scaled.shape)
print('X_test_scaled: ', X_test_scaled.shape)
print('Mean check (should be ~0):', X_train_scaled.mean().abs().mean().round(6))
print('Std  check (should be ~1):', X_train_scaled.std().mean().round(6))

In [ ]:
# Distribution before vs after scaling for one feature
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
X_train_eng['rssi1'].hist(bins=40, ax=axes[0], color='steelblue')
axes[0].set_title('rssi1 — before scaling')
X_train_scaled['rssi1'].hist(bins=40, ax=axes[1], color='coral')
axes[1].set_title('rssi1 — after StandardScaler')
plt.tight_layout()
plt.savefig('fig_scaling_example.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Also save raw IQ version (without engineering)

Some models (e.g. tree-based: Random Forest, XGBoost) do not require normalization and may even perform better on raw IQ features. We save both versions.

In [ ]:
# Raw IQ (guard subcarriers and seq_ctrl already dropped), NOT scaled
X_train_raw_clean = X_train_clean.copy()
X_test_raw_clean  = X_test_clean.copy()

scaler_raw = StandardScaler()
X_train_raw_scaled = pd.DataFrame(
    scaler_raw.fit_transform(X_train_raw_clean),
    columns=X_train_raw_clean.columns,
    index=X_train_raw_clean.index
)
X_test_raw_scaled = pd.DataFrame(
    scaler_raw.transform(X_test_raw_clean),
    columns=X_test_raw_clean.columns,
    index=X_test_raw_clean.index
)
print('Raw IQ scaled - train:', X_train_raw_scaled.shape)
print('Raw IQ scaled - test: ', X_test_raw_scaled.shape)

## 10. Saving preprocessed data

In [ ]:
# Save engineered + scaled datasets (main version — for SVM, KNN, LR, MLP)
X_train_scaled.to_csv('X_train_eng_scaled.csv')
X_test_scaled.to_csv('X_test_eng_scaled.csv')
y_train.to_csv('y_train.csv', header=True)

# Save raw IQ + scaled (for tree-based models)
X_train_raw_scaled.to_csv('X_train_raw_scaled.csv')
X_test_raw_scaled.to_csv('X_test_raw_scaled.csv')

# Save engineered + NOT scaled (for Random Forest / XGBoost)
X_train_eng.to_csv('X_train_eng.csv')
X_test_eng.to_csv('X_test_eng.csv')

# Save the scaler and feature names for later use in other notebooks
with open('scaler_eng.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('scaler_raw.pkl', 'wb') as f:
    pickle.dump(scaler_raw, f)
with open('active_subcarriers.pkl', 'wb') as f:
    pickle.dump(active_sub, f)

print('All files saved successfully.')
print()
print('Files generated:')
print('  X_train_eng_scaled.csv  — engineered features, scaled   → use with SVM/KNN/LR')
print('  X_test_eng_scaled.csv')
print('  X_train_raw_scaled.csv  — raw I/Q (clean), scaled       → alternative baseline')
print('  X_test_raw_scaled.csv')
print('  X_train_eng.csv         — engineered features, unscaled → use with RF/XGBoost')
print('  X_test_eng.csv')
print('  y_train.csv             — labels (0-9)')
print('  scaler_eng.pkl          — fitted StandardScaler (eng features)')
print('  scaler_raw.pkl          — fitted StandardScaler (raw IQ)')
print('  active_subcarriers.pkl  — list of active subcarrier indices')

## 11. Summary

| Step | Input cols | Output cols | Notes |
|---|---|---|---|
| Raw data | 261 | 261 | Including label |
| Drop seq_ctrl + guard subcarriers | 261 | 211 | DC + guard band (indices 0, 27–37) |
| Feature engineering | 211 | 315 | amp, phase, dphi, aratio per active subcarrier |
| StandardScaler | 315 | 315 | Fit on train only |

**Key findings from EDA:**
- No missing values; dataset is clean.
- Classes are well balanced (1200–1488 samples each) — no need for class imbalance techniques.
- RSSI clearly separates 2 m positions (0–4) from 5 m positions (5–9).
- Inter-antenna phase difference (`dphi`) varies consistently per position, confirming angle-of-arrival discrimination.
- Amplitude profiles across subcarriers are distinct per position (Fig. 1 of report).
- `aoa` has very high variance and overlapping class distributions — useful as a feature but not alone sufficient.

**Next steps:** Use these preprocessed files in the classification notebooks (SVM, Random Forest, KNN, etc.).